# Interactive Signal Processing Demo
## Fourier Series Classification: Signal Generation, Fourier Analysis, and Jump Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbass12/FourierSeriesClassification/blob/main/notebooks/Interactive_Signal_Demo.ipynb)

This notebook provides interactive visualizations of:
1. Signal generation with configurable parameters
2. Fourier series approximation at different mode counts
3. Edge detection using concentration factors (Gelb-Tadmor method)
4. Neural network classification comparison

In [ ]:
# Bootstrap the repository when opened directly in Google Colab.
import os, sys
from pathlib import Path

if not Path('src').exists() and not Path('../src').exists():
    !git clone --depth 1 https://github.com/abbass12/FourierSeriesClassification.git
    os.chdir('FourierSeriesClassification')

!pip install -q -r requirements.txt ipywidgets
root = Path('.') if Path('src').exists() else Path('..')
sys.path.insert(0, str(root / 'src'))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

from signals import *
from fourier import *

## 1. Interactive Signal Generation
Explore the five signal types with adjustable parameters.

In [ ]:
x = generate_grid(1500)

def plot_signal(signal_type='box', a=1.5, b=2.0, snr=30):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    
    # Generate signal
    gen = SIGNAL_GENERATORS[signal_type]
    if signal_type in ['sine', 'exponential']:
        signal = gen(x, a=a, b=b)
    elif signal_type == 'gaussian':
        signal = gen(x, a=a, b=int(b))
    else:
        signal = gen(x, a=a, b=b)
    
    noisy = add_noise(signal, snr)
    
    # Plot clean signal
    axes[0].plot(x, signal, 'b-', linewidth=1.5)
    axes[0].set_title(f'{signal_type.capitalize()} (clean)')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([-np.pi, np.pi])
    
    # Plot noisy signal
    axes[1].plot(x, noisy, 'r-', linewidth=0.5, alpha=0.7)
    axes[1].set_title(f'{signal_type.capitalize()} (SNR={snr}dB)')
    axes[1].set_xlabel('x')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([-np.pi, np.pi])
    
    # Plot Fourier coefficients
    coeffs = compute_fourier_coefficients(signal, 100)
    k = np.arange(-50, 50)
    axes[2].semilogy(k, np.abs(coeffs) + 1e-16, 'g.-', markersize=3)
    axes[2].set_title('|Fourier Coefficients|')
    axes[2].set_xlabel('Mode k')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

interact(plot_signal,
         signal_type=Dropdown(options=SIGNAL_NAMES, value='box'),
         a=FloatSlider(min=0.5, max=2.5, step=0.1, value=1.5),
         b=FloatSlider(min=0.5, max=5.0, step=0.5, value=2.0),
         snr=IntSlider(min=5, max=40, step=5, value=20));

## 2. Fourier Series Approximation
See how increasing the number of Fourier modes improves the approximation.

In [ ]:
def plot_fourier_approx(signal_type='box', n_modes=20):
    gen = SIGNAL_GENERATORS[signal_type]
    signal = gen(x)
    coeffs = compute_fourier_coefficients(signal, n_modes)
    reconstruction = fourier_partial_sum(coeffs, x)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(x, signal, 'b-', linewidth=1.5, label='Original', alpha=0.6)
    axes[0].plot(x, reconstruction, 'r-', linewidth=1.5, label=f'N={n_modes} modes')
    axes[0].set_title(f'Fourier Approximation (N={n_modes})')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Amplitude')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([-np.pi, np.pi])
    
    # Error plot
    error = np.abs(signal - reconstruction)
    axes[1].plot(x, error, 'k-', linewidth=1.0)
    axes[1].set_title(f'Approximation Error (max={np.max(error):.4f})')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('|Error|')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([-np.pi, np.pi])
    
    plt.tight_layout()
    plt.show()

interact(plot_fourier_approx,
         signal_type=Dropdown(options=SIGNAL_NAMES, value='box'),
         n_modes=IntSlider(min=3, max=200, step=1, value=20));

## 3. Edge Detection via Concentration Factors
Visualize how the Gelb-Tadmor method detects jump discontinuities from Fourier data.

In [ ]:
def plot_edge_detection(signal_type='box', n_modes=50, sigma_type='trig'):
    gen = SIGNAL_GENERATORS[signal_type]
    signal = gen(x)
    coeffs = compute_fourier_coefficients(signal, n_modes)
    edge_fn = generalized_conjugate_partial_sum(coeffs, x, sigma_type)
    
    # Detect jumps
    locations, values = detect_jumps(edge_fn, x)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(x, signal, 'b-', linewidth=1.5)
    axes[0].set_title(f'{signal_type.capitalize()} Signal')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim([-np.pi, np.pi])
    
    # Mark detected jumps on signal
    for loc in locations:
        axes[0].axvline(x=loc, color='r', linestyle='--', alpha=0.7)
    
    axes[1].plot(x, edge_fn, 'r-', linewidth=1.0)
    axes[1].axhline(y=0, color='k', linewidth=0.5)
    axes[1].set_title(f'Edge Function ({sigma_type} sigma, N={n_modes})')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('[f](x)')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim([-np.pi, np.pi])
    
    # Mark detected jumps
    for loc, val in zip(locations, values):
        axes[1].plot(loc, val, 'go', markersize=10)
    
    if len(locations) > 0:
        print(f'Detected {len(locations)} jump(s):')
        for i, (loc, val) in enumerate(zip(locations, values)):
            print(f'  Jump {i+1}: location={loc:.4f}, magnitude={val:.4f}')
    else:
        print('No jumps detected (smooth signal)')
    
    plt.tight_layout()
    plt.show()

interact(plot_edge_detection,
         signal_type=Dropdown(options=SIGNAL_NAMES, value='box'),
         n_modes=IntSlider(min=10, max=200, step=5, value=50),
         sigma_type=Dropdown(options=['trig', 'poly', 'exp'], value='trig'));

## 4. Classification Results Summary
Compare the performance of Models A, B, and C.

In [ ]:
import json

# Load results (if available)
try:
    with open('../results/experiment_results.json') as f:
        results = json.load(f)
    
    print('=== Classification Results ===')
    print(f"Model A (Raw signals, clean):  {results['model_a']['clean']['accuracy']:.4f}")
    print(f"Model B (Fourier, N=50):       {results['model_b']['N_50']['accuracy']:.4f}")
    print(f"Model C (Fourier+Jumps, N=50): {results['model_c']['N_50']['accuracy']:.4f}")
    
    print('\n=== Model B: Accuracy vs N ===')
    for key, val in sorted(results['model_b'].items()):
        print(f"  {key}: {val['accuracy']:.4f}")
    
    print('\n=== Model C: Accuracy vs N ===')
    for key, val in sorted(results['model_c'].items()):
        print(f"  {key}: {val['accuracy']:.4f}")
        
except FileNotFoundError:
    print('Results file not found. Run the experiments first:')
    print('  python src/run_experiments.py')